使用服务器的qwen环境运行 多路召回（文本+向量）

In [3]:
# 从 langchain_community 中导入 BM25Retriever，基于 BM25 算法的检索器（传统 IR 方法，非向量）
from langchain_community.retrievers import BM25Retriever

# 导入类型注解工具，用于函数参数或返回值的类型说明（List 等）
from typing import List

# 导入结巴分词，用于中文文本的分词处理
import jieba

# 文本切分工具，递归地按字符长度切分大段文本（适合做向量化输入）
from langchain.text_splitter import RecursiveCharacterTextSplitter

# 导入 FAISS 向量数据库，用于存储和检索文本向量
from langchain.vectorstores import FAISS

# 导入文本加载器，可以从 txt 文件中加载文本到 LangChain Document 对象
from langchain.document_loaders import TextLoader

# 导入 HuggingFaceEmbeddings，用于加载 HuggingFace 的文本向量模型
from langchain_huggingface import HuggingFaceEmbeddings


In [4]:
loader = TextLoader('medical_data.txt')

# 调用 load() 方法，把文本文件内容加载为 LangChain 的 Document 对象列表
documents = loader.load()

# 2. 定义文本切分器
# RecursiveCharacterTextSplitter 会把长文本拆分成小块（chunk），方便后续做向量化和检索
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,       # 每个文本块的最大长度为 500 字符
    chunk_overlap  = 0,     # 相邻的文本块之间不重叠
    length_function = len,  # 用 Python 内置的 len 函数来计算长度
    separators=['\n']       # 优先按照换行符切分
)

# 3. 执行文本切分
# 把原始文档列表分割成多个小片段（docs），每个片段仍然是 Document 类型
docs = text_splitter.split_documents(documents)

In [5]:
docs[0]

Document(metadata={'source': 'medical_data.txt'}, page_content="{'question': '曲匹地尔片的用法用量', 'answer': '注意：同种药品可由于不同的包装规格有不同的用法或用量。本文只供参考。如果不确定，请参看药品随带的说明书或向医生询问。口服。一次50～100mg（1-2片），3次/日，或遵医嘱。'}")

In [6]:
docs[1]

Document(metadata={'source': 'medical_data.txt'}, page_content="\n{'question': '三期梅毒多久能治愈吗', 'answer': '梅毒是一种进展十分缓慢的疾病，包括早期梅毒，晚期梅毒等等，一般来说，梅毒病程超过两年的就是晚期梅毒了，而晚期梅毒的治疗难度是比较大的，三期梅毒能完全治愈吗?该如何进行治疗?一、对于三期梅毒的治疗，中西各有妙招，希望患者在医生的指导下选择适合自己的治疗方法。被确诊为三期梅毒的患者很关心这个问题，梅毒需要讲究科学的治疗方法，三期梅毒患者如果有一个良好的心态，积极配合医生治疗，还有很有可能治愈的，具体治疗方法我们看看下文的介绍。二、三期梅毒又称晚期梅毒，是因梅毒初期没有进行治疗或者治疗不彻底所造成。好发于40-50岁之间。该期梅毒不仅局限于皮肤和粘膜，也可侵犯任何内脏器官和组织。三期梅毒可发生在感染后2年以上，一般多发于感染后3～4年。病程漫长，可持续10～30年。未经治愈的二期梅毒中约有1/3的病人可发展为晚期活动性梅毒;另有一部分患者不出现晚期梅毒症状，只是梅毒血清反应持续阳性，为晚期潜伏梅毒;也有一部分患者可以自愈。三、晚期梅毒可以通过药物治疗控制病情，但是治愈的可能性是比较小的，治疗还是有必要的，通过有效的治疗可以缓解患者的症状，杀灭一部分的梅毒螺旋体。虽然治愈的可能性不高，但是患者还是应当坚持治疗。通过上文的简单介绍，相信大家对于三期梅毒多久能治愈这一方面的问题已经有了自己的答案。对于梅毒患者来说，无论患病多少年，都应在确诊后及时进行正规的治疗处理。正规的治疗可以缓解患者的病情，患者还是应当坚持进行，并积极配合医生的治疗，说不定治愈的可能就发生了。'}")

In [ ]:
# 定义分词预处理函数
# 输入：一段文本字符串
# 输出：用 jieba 分词后的词列表
def preprocessing_func(text: str) -> List[str]:
    return list(jieba.cut(text))


# 源码前处理函数就是空格划分 默认四条
# bm25 = BM25Retriever(docs=docs,k=10)
# print(bm25.k)

# # 这样检索时会先对输入文本和文档进行分词，再计算 BM25 相似度
# retriever = bm25.from_documents(docs,preprocess_func=preprocessing_func)


# 修改后支持10个结果
retriever = BM25Retriever.from_documents(
    docs,
    preprocess_func=preprocessing_func,
    k=10
)

In [ ]:
result = retriever.invoke('骨折了应该怎么办')
print(result)
print(len(result))


[Document(metadata={'source': 'medical_data.txt'}, page_content="\n{'question': '手指骨折后弯不了怎么办', 'answer': '手指骨骨折是指手部的骨折。较其他部位骨折多见，且多为多发性骨折，骨折后有典型的移位和畸形，诊断比较容易，但仍需行X线摄片，详细了解其移位情况，从而用不同的手法进行复位和固定。手指骨折后弯不了怎么办手指骨折的治疗曾因对位不佳或固定不牢固,而产生畸形愈合或者不愈合，也常因固定不当或固定时间过长而至关节囊或侧副韧带挛缩，导致关节僵硬，特别是关节附近和颈关节的骨折，常导致关节强直，严重影响手指的功能，所以手指骨折的关键在于早期的复位固定，一般错位明显的需要手术治疗，对位良好的可以保守，石膏托外固定治疗，一般恢复需要3个月左右的时间，治疗期间要逐渐进行关节功能恢复训练，预防关节僵硬。骨折后的1-2周是第一阶段，骨折部位肿胀并且有大面积淤血，经络不通，气血阻滞，此时的饮食应以活血化瘀、行气消散为主，饮食以清淡为主如蔬菜、水果、蛋类、肉类、瘦肉，忌食辛辣油腻的食物；骨折后的2-4周是第二阶段，也最为主要的阶段，症状已消退骨骼已开始恢复。饮食要保持足够的营养，满足骨骼生长需要，可以多吃富含钙质和维生素的食品，如纯牛奶、鸡蛋、豆制品、瘦肉、青菜、萝卜等，从而可以促进骨痂生长和伤口的恢复；骨折后4-8周是第三阶段，此时骨折基本愈合脉络通畅，骨骼恢复进入到最后阶段。饮食方面要以保健预防为主，可以促进牢固的骨痂生长并且使舒筋活络，使骨折部的邻近关节能够自由的活动，从而恢复往日的功能。可以吃鸡汤、羊骨汤、鱼汤等，而且饮食方面的禁忌也逐渐减少。'}"), Document(metadata={'source': 'medical_data.txt'}, page_content="\n{'question': '手腕骨折一直肿胀怎么办', 'answer': '手腕骨折一直肿胀被认为是血液循环不良的原因。建议首先注意休息，避免过度活动，尤其是手腕活动。白天，受影响的上肢可以挂在脖子上。每天局部敷热敷。干燥后，尝试外部消毒。同时，你可以饮用活血化瘀的药物。如果是风湿性疾病，要根据检查结果选择治疗合适的治疗方法的。手腕骨折在日常生活中较为常见，大部分受伤者是老年人，大部分患者是由于手掌落地

In [29]:
retriever.get_relevant_documents('冲冲悄咪咪cznzcn啊啊啊啊啊啊啊')


[Document(metadata={'source': 'medical_data.txt'}, page_content="\n{'question': '男生痘痘肌肤怎么改善', 'answer': '男生改善痘痘肌首先要注意油性皮肤选择护肤产品一定不能是油腻的，应该先要滋润皮肤，保护皮肤表层，然后再呵护皮肤，日常洁面时要用温水洁面，帮助打开毛孔，然后用清洁力度适中的洁面乳清洗，一定要注意自己的饮食习惯，少吃辛辣的食物，多吃清淡一点的食物，这样可以改善自己的肠胃，对于痘痘的皮肤也有很好的改善，缓解痘痘的肌肤也要适量的做一下运动。男生改善痘痘肌肤首先，油性皮肤选择护肤产品一定不能是油腻的。应该先要滋润皮肤，保护皮肤表层，然后再呵护皮肤。可以选涂抹上一层清爽的水乳，为皮肤起到保护、抑制油脂分泌的作用。建议在夏秋季节可以选择含水量较高的爽肤水和乳液，进入到春季，油皮可选择温和的清爽型产品，干皮可适当加入清爽型的面霜啫喱。男生改善痘痘肌肤日常洁面时要用温水洁面，帮助打开毛孔，然后用清洁力度适中的洁面乳清洗，用洁面乳洗脸的时候要轻柔地按摩脸部，更深层地清除毛孔中的污垢，这样你的皮肤就不会因为毛孔堵塞而长出痘痘，也就不会形成痘痘肌。而痘痘肌也会有所改善。为了改善自己的痘痘肌肤，就一定要注意自己的饮食习惯，通常一些那些好吃的无非就是辛辣的食物，这些就不要在吃啦，因为吃这些只会加重自己的痘痘症状，这样对于自己的护肤是没有一点的帮助，所以一般都是吃些水果啊，蔬菜啊，其他就是一些清淡一点的食物，这样可以改善自己的肠胃，对于痘痘的皮肤也有很好的改善。其实男生想要缓解痘痘的肌肤也要适量的做一下运动，这样可以保证机体的功能正常的运作，可以促进血液的循环速度，帮助代谢的正常进行，都是可以缓解痘痘的症状，而且运动的话可以适当出些汗，可以及时清理毛孔，是很有帮助。'}"),
 Document(metadata={'source': 'medical_data.txt'}, page_content="\n{'question': '子宫肌层欠均什么意思', 'answer': '子宫肌层是指构成哺乳类子宫中层的平滑肌组织。卵巢分泌的性激素能刺激子宫平滑肌纤维，而引起与性周期相应的肥大和增生。另外，妊娠时子宫肌层的平滑肌纤维，可肥大到哺乳类平滑肌的最大。分娩时子宫肌能对脑垂体后叶激素的催产

In [ ]:
# up自己实现的
from rank_bm25 import BM25Okapi
texts = [i.page_content for i in docs]
texts_processed = [preprocessing_func(t) for t in texts]
vectorizer = BM25Okapi(texts_processed)

In [31]:
vectorizer.get_top_n(preprocessing_func('骨折了应该怎么办'),texts, n=10)

["\n{'question': '手指骨折后弯不了怎么办', 'answer': '手指骨骨折是指手部的骨折。较其他部位骨折多见，且多为多发性骨折，骨折后有典型的移位和畸形，诊断比较容易，但仍需行X线摄片，详细了解其移位情况，从而用不同的手法进行复位和固定。手指骨折后弯不了怎么办手指骨折的治疗曾因对位不佳或固定不牢固,而产生畸形愈合或者不愈合，也常因固定不当或固定时间过长而至关节囊或侧副韧带挛缩，导致关节僵硬，特别是关节附近和颈关节的骨折，常导致关节强直，严重影响手指的功能，所以手指骨折的关键在于早期的复位固定，一般错位明显的需要手术治疗，对位良好的可以保守，石膏托外固定治疗，一般恢复需要3个月左右的时间，治疗期间要逐渐进行关节功能恢复训练，预防关节僵硬。骨折后的1-2周是第一阶段，骨折部位肿胀并且有大面积淤血，经络不通，气血阻滞，此时的饮食应以活血化瘀、行气消散为主，饮食以清淡为主如蔬菜、水果、蛋类、肉类、瘦肉，忌食辛辣油腻的食物；骨折后的2-4周是第二阶段，也最为主要的阶段，症状已消退骨骼已开始恢复。饮食要保持足够的营养，满足骨骼生长需要，可以多吃富含钙质和维生素的食品，如纯牛奶、鸡蛋、豆制品、瘦肉、青菜、萝卜等，从而可以促进骨痂生长和伤口的恢复；骨折后4-8周是第三阶段，此时骨折基本愈合脉络通畅，骨骼恢复进入到最后阶段。饮食方面要以保健预防为主，可以促进牢固的骨痂生长并且使舒筋活络，使骨折部的邻近关节能够自由的活动，从而恢复往日的功能。可以吃鸡汤、羊骨汤、鱼汤等，而且饮食方面的禁忌也逐渐减少。'}",
 "\n{'question': '手腕骨折一直肿胀怎么办', 'answer': '手腕骨折一直肿胀被认为是血液循环不良的原因。建议首先注意休息，避免过度活动，尤其是手腕活动。白天，受影响的上肢可以挂在脖子上。每天局部敷热敷。干燥后，尝试外部消毒。同时，你可以饮用活血化瘀的药物。如果是风湿性疾病，要根据检查结果选择治疗合适的治疗方法的。手腕骨折在日常生活中较为常见，大部分受伤者是老年人，大部分患者是由于手掌落地后摔倒造成的。大多数骨折发生在桡骨远端关节附近2厘米处，临床上称为科利尔骨折。没有及时有效的治疗，患者的腕关节会出现关节畸形、关节运动功能受限、关节疼痛等症状，给患者的日常生活带来诸多不便。因此，在诊断和治疗中必须达到正确的复位和良好的固定。骨折初期肿

In [32]:
import os
os.environ["http_proxy"] = "http://127.0.0.1:7890"
os.environ["https_proxy"] = "http://127.0.0.1:7890"
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-zh-v1.5", model_kwargs = {'device': 'cuda:1'})

# from langchain.embeddings import HuggingFaceBgeEmbeddings
# model_name = "BAAI/bge-large-zh-v1.5"
# model_kwargs = {'device': 'cuda'}
# encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
# model = HuggingFaceBgeEmbeddings(
#     model_name=model_name,
#     model_kwargs=model_kwargs,
#     encode_kwargs=encode_kwargs,
#     query_instruction="为这个句子生成表示以用于检索相关文章："
# )
# model.query_instruction = "为这个句子生成表示以用于检索相关文章："


/data/ljr/anaconda3/envs/qwen/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/data/ljr/anaconda3/envs/qwen/lib/python3.10/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [34]:
# 向量召回 直接把文档列表转成 FAISS 向量索引。

# 内部流程大概是：

# 遍历 docs，调用 embeddings 把每条文本转成向量。

# 用 FAISS 建立索引，把向量和原文档对应起来。

# 返回一个 FAISS 对象 db，你可以用它做向量检索。
db = FAISS.from_documents(docs, embeddings)

In [36]:
db.save_local('faiss_index')

In [38]:
# 加载本地 FAISS 索引
db = FAISS.load_local(
    folder_path='faiss_index',
    embeddings=embeddings,
    allow_dangerous_deserialization=True  # ⚠️ 允许加载 pickle 文件
)

两类方法返回的前10相似结果

In [39]:
bm25_res = vectorizer.get_top_n(preprocessing_func('骨折了应该怎么办'),texts, n=10)
bm25_res

["\n{'question': '手指骨折后弯不了怎么办', 'answer': '手指骨骨折是指手部的骨折。较其他部位骨折多见，且多为多发性骨折，骨折后有典型的移位和畸形，诊断比较容易，但仍需行X线摄片，详细了解其移位情况，从而用不同的手法进行复位和固定。手指骨折后弯不了怎么办手指骨折的治疗曾因对位不佳或固定不牢固,而产生畸形愈合或者不愈合，也常因固定不当或固定时间过长而至关节囊或侧副韧带挛缩，导致关节僵硬，特别是关节附近和颈关节的骨折，常导致关节强直，严重影响手指的功能，所以手指骨折的关键在于早期的复位固定，一般错位明显的需要手术治疗，对位良好的可以保守，石膏托外固定治疗，一般恢复需要3个月左右的时间，治疗期间要逐渐进行关节功能恢复训练，预防关节僵硬。骨折后的1-2周是第一阶段，骨折部位肿胀并且有大面积淤血，经络不通，气血阻滞，此时的饮食应以活血化瘀、行气消散为主，饮食以清淡为主如蔬菜、水果、蛋类、肉类、瘦肉，忌食辛辣油腻的食物；骨折后的2-4周是第二阶段，也最为主要的阶段，症状已消退骨骼已开始恢复。饮食要保持足够的营养，满足骨骼生长需要，可以多吃富含钙质和维生素的食品，如纯牛奶、鸡蛋、豆制品、瘦肉、青菜、萝卜等，从而可以促进骨痂生长和伤口的恢复；骨折后4-8周是第三阶段，此时骨折基本愈合脉络通畅，骨骼恢复进入到最后阶段。饮食方面要以保健预防为主，可以促进牢固的骨痂生长并且使舒筋活络，使骨折部的邻近关节能够自由的活动，从而恢复往日的功能。可以吃鸡汤、羊骨汤、鱼汤等，而且饮食方面的禁忌也逐渐减少。'}",
 "\n{'question': '手腕骨折一直肿胀怎么办', 'answer': '手腕骨折一直肿胀被认为是血液循环不良的原因。建议首先注意休息，避免过度活动，尤其是手腕活动。白天，受影响的上肢可以挂在脖子上。每天局部敷热敷。干燥后，尝试外部消毒。同时，你可以饮用活血化瘀的药物。如果是风湿性疾病，要根据检查结果选择治疗合适的治疗方法的。手腕骨折在日常生活中较为常见，大部分受伤者是老年人，大部分患者是由于手掌落地后摔倒造成的。大多数骨折发生在桡骨远端关节附近2厘米处，临床上称为科利尔骨折。没有及时有效的治疗，患者的腕关节会出现关节畸形、关节运动功能受限、关节疼痛等症状，给患者的日常生活带来诸多不便。因此，在诊断和治疗中必须达到正确的复位和良好的固定。骨折初期肿

In [40]:
vector_res = db.similarity_search('骨折了应该怎么办', k=10)
vector_res

[Document(id='38b54dcf-c8a2-4b4a-96a5-dbb799ff3cf5', metadata={'source': 'medical_data.txt'}, page_content="\n{'question': '骨折术后护理', 'answer': '骨折病人术后护理要做好外固定夹板，石膏，支具的护理。要观察外固定的松紧度，骨折早期可能因整复后1到3天，肢体明显肿胀外固定太紧，要注意观察末梢血液循环的情况，及时调整绷带的松紧度一面太紧而引起肢体缺血坏死。骨折中后期因肿胀消退后，外固定可能会松动，所以必须定期复查，以免造成骨折断端移位。如果是有局部外伤引起了局部的剧烈疼痛畸形一般是有骨折了。外伤后应该首先充分休息制动，避免小腿再活动。尽快做片子检查。。如果拍片检查确定是骨折应该尽快局部外固定充分制动，避免骨折断端的活动引起出血水肿加重或引起血管神经的损伤。完全固定后消肿治疗。同时确定是否需要进一步手术固定治疗。住院环境和作息制度给病人制造一个安静、安全、舒适、卫生的住院环境，保持室内空气新鲜，帮助病人制定规律生活和作息制度，保证充足的睡眠。这也是骨折的术后护理措施。病情观察由于老年人体内器官，代偿能力差，机体易出现电解质紊乱、全身衰竭等并发症，所以老年骨折病人在手术时机体受到极大刺激后，尤其要注意生命体征及全身情况的观察，防止因骨折手术加重老年骨折病人原有的疾患。骨折的前期阶段。骨折患者这个时候骨折后刚经过治疗，骨折部位还是肿胀，有大片的淤血，经络不通，气血阻滞，这个时候的饮食应以活血化瘀，行气消散为主。饮食以清淡为主，如蔬菜、蛋类、水果、瘦肉等，忌食辛辣、燥热、油腻。这样有利于淤血的退消，脉络的通畅，比较有利于骨骼的愈合。有利于关节共能的恢复。最主要的阶段，这时患者症状基本都已经退化。骨骼开始愈合，饮食要以营养为主，以满足骨骼生长需要，促进骨骼的愈合。多吃些骨头汤，动物的肝脏，多补充一些维生素a、d、钙已及蛋白质。吃些青菜、包菜、萝卜等维生素c含量丰富的蔬菜，以促进骨痂生长和伤口愈合。'}"),
 Document(id='f55ee184-9b21-453b-8645-b00b47376c3b', metadata={'source': 'medical_data.txt'}, page_content="{'question'

In [41]:
def rrf(vector_results: List[str], text_results: List[str], k: int=10, m: int=60):
        """
        使用RRF算法对两组检索结果进行重排序
        
        params:
        vector_results (list): 向量召回的结果列表,每个元素是专利ID
        text_results (list): 文本召回的结果列表,每个元素是专利ID
        k(int): 排序后返回前k个
        m (int): 超参数
        
        return:
        重排序后的结果列表,每个元素是(文档ID, 融合分数)
        """
        
        doc_scores = {}
        
        # 遍历两组结果,计算每个文档的融合分数
        # 公式: score = 1 / (rank + m)
        # rank 从 0 开始, m 是平滑参数，防止分母过小
        for rank, doc_id in enumerate(vector_results):
            doc_scores[doc_id] = doc_scores.get(doc_id, 0) + 1 / (rank+m)
        for rank, doc_id in enumerate(text_results):
            doc_scores[doc_id] = doc_scores.get(doc_id, 0) + 1 / (rank+m)
        
        # 将结果按融合分数排序 按融合分数从高到低排序，并取前 k 个
        sorted_results = [d for d, _ in sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:k]]

        return sorted_results

In [42]:
vector_results = [i.page_content for i in vector_res]
text_results = [i for i in bm25_res]
rrf_res = rrf(vector_results, text_results)
rrf_res

["\n{'question': '手指骨折后弯不了怎么办', 'answer': '手指骨骨折是指手部的骨折。较其他部位骨折多见，且多为多发性骨折，骨折后有典型的移位和畸形，诊断比较容易，但仍需行X线摄片，详细了解其移位情况，从而用不同的手法进行复位和固定。手指骨折后弯不了怎么办手指骨折的治疗曾因对位不佳或固定不牢固,而产生畸形愈合或者不愈合，也常因固定不当或固定时间过长而至关节囊或侧副韧带挛缩，导致关节僵硬，特别是关节附近和颈关节的骨折，常导致关节强直，严重影响手指的功能，所以手指骨折的关键在于早期的复位固定，一般错位明显的需要手术治疗，对位良好的可以保守，石膏托外固定治疗，一般恢复需要3个月左右的时间，治疗期间要逐渐进行关节功能恢复训练，预防关节僵硬。骨折后的1-2周是第一阶段，骨折部位肿胀并且有大面积淤血，经络不通，气血阻滞，此时的饮食应以活血化瘀、行气消散为主，饮食以清淡为主如蔬菜、水果、蛋类、肉类、瘦肉，忌食辛辣油腻的食物；骨折后的2-4周是第二阶段，也最为主要的阶段，症状已消退骨骼已开始恢复。饮食要保持足够的营养，满足骨骼生长需要，可以多吃富含钙质和维生素的食品，如纯牛奶、鸡蛋、豆制品、瘦肉、青菜、萝卜等，从而可以促进骨痂生长和伤口的恢复；骨折后4-8周是第三阶段，此时骨折基本愈合脉络通畅，骨骼恢复进入到最后阶段。饮食方面要以保健预防为主，可以促进牢固的骨痂生长并且使舒筋活络，使骨折部的邻近关节能够自由的活动，从而恢复往日的功能。可以吃鸡汤、羊骨汤、鱼汤等，而且饮食方面的禁忌也逐渐减少。'}",
 "\n{'question': '骨折术后护理', 'answer': '骨折病人术后护理要做好外固定夹板，石膏，支具的护理。要观察外固定的松紧度，骨折早期可能因整复后1到3天，肢体明显肿胀外固定太紧，要注意观察末梢血液循环的情况，及时调整绷带的松紧度一面太紧而引起肢体缺血坏死。骨折中后期因肿胀消退后，外固定可能会松动，所以必须定期复查，以免造成骨折断端移位。如果是有局部外伤引起了局部的剧烈疼痛畸形一般是有骨折了。外伤后应该首先充分休息制动，避免小腿再活动。尽快做片子检查。。如果拍片检查确定是骨折应该尽快局部外固定充分制动，避免骨折断端的活动引起出血水肿加重或引起血管神经的损伤。完全固定后消肿治疗。同时确定是否需要进一步手术固定治疗。住院环境和作息制度给病人

In [43]:
prompt = '''
任务目标：根据检索出的文档回答用户问题
任务要求：
    1、不得脱离检索出的文档回答问题
    2、若检索出的文档不包含用户问题的答案，请回答我不知道

用户问题：
{}

检索出的文档：
{}
'''

In [44]:
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model='Qwen2-7B-Instruct', base_url='http://localhost:1234/v1', api_key='n')
res = model.invoke(prompt.format('骨折了应该怎么办', ''.join(rrf_res)))
print(res.content)

关于骨折后的处理和恢复问题，文档中提供了多种信息来回答您的问题：

1. **手指骨折后弯不了怎么办**：需要通过正确的手法复位，并使用铝板或石膏固定，通常在4周后进行功能锻炼。

2. **骨折术后护理**：包括观察外固定的松紧度、饮食调整（早期以活血化瘀、行气消散为主，中期和后期则需满足骨骼生长的营养需求），以及保持室内环境安静舒适，确保充足的睡眠。

3. **脚趾骨折吃什么水果好**：建议吃香蕉、橙子等富含维生素C的水果，因为这些水果有助于促进骨折愈合。同时，多吃新鲜蔬菜以避免便秘。

4. **手腕骨折一直肿胀怎么办**：需要休息、减少活动，并定期热敷和冷敷以帮助消肿，确保血液循环良好。

5. **骨盆骨折的并发症是什么**：可能包括腹膜后血肿、尿道或膀胱损伤、直肠损伤以及神经损伤等。

6. **锁骨骨折多久能干活**：通常需要3个月以上的时间，具体视个人恢复情况而定，初期应避免重体力劳动。

7. **手指指骨骨折多久愈合**：根据损伤程度不同，一般需要4-5个月左右的恢复期，但具体时间还需根据医生的专业指导来确定。

8. **跖骨骨折吃什么水果好**：猕猴桃、番石榴、草莓、柿子和柑橘等富含维生素C的水果有利于骨折后的康复。

在处理骨折问题时，请确保遵循专业医疗人员的建议，并根据个人情况调整饮食和活动计划。


In [45]:
res = model.invoke('骨折了应该怎么办')
print(res.content)

如果怀疑自己或他人有骨折，应遵循以下步骤处理：

1. **保持冷静**：首先确保现场安全，避免在移动受伤者时造成二次伤害。

2. **检查呼吸和循环**：确认伤者是否有生命体征（呼吸、脉搏）。如果有严重出血或其他紧急情况，立即拨打急救电话，并进行初步的紧急救护。

3. **固定伤处**：使用夹板或硬物作为临时固定器材。在没有适当的固定材料的情况下，可以用毯子、衣服等物品包裹受伤部位，切忌尝试活动已骨折的部位以避免加重伤害。

4. **保持姿势稳定**：将伤者平躺在柔软但支撑力足够的地方，并用枕头或其他软物体垫高脚部（适用于腿部骨折），以减轻疼痛和肿胀。如果可能，尽量让伤者的四肢与躯干保持水平或稍高于心脏位置。

5. **寻求专业医疗帮助**：立即联系急救服务，告知情况的严重性并等待救援人员到来。在等待期间，不要尝试自行移动或复位骨折部位，以免引起更严重的伤害。

6. **保持通讯畅通**：确保伤者周围有至少一个成人能够提供信息给救援人员，并保持与医疗团队的有效沟通。

记住，在处理任何疑似骨折的情况时，最重要的是避免自行尝试移动或固定受伤的部位，应由专业医疗人员进行操作。骨折通常需要专业的诊断和治疗，包括可能的手术、石膏固定或其他治疗方法。
